In [ ]:
%load ../lib/RecursiveSet.jsh
%load ConstraintCompiler.jsh
%load 02-Backtracking-Constraint-Solver.jsh

The function `range(start, stop)` returns the list `[start, ..., stop]`.

In [ ]:
List<Integer> range(int start, int stop) {
    return IntStream.rangeClosed(start, stop).boxed().toList();
}

# The 8-Queens Problem

The <a href="https://en.wikipedia.org/wiki/Eight_queens_puzzle">eight queens puzzle</a> is the problem of placing eight chess queens on a chessboard so that no two queens can capture each other.  In <a href="https://en.wikipedia.org/wiki/Chess">chess</a> a queen can capture another piece if this piece is either
<ol>
    <li>in the same row,</li>
    <li>in the same column, or</li>
    <li>in the same diagonal.</li>
</ol>
The image below shows a queen in row 3, column 4.  All the locations where a piece can be captured by this queen are marked with an arrow.

<img src="queen-captures.png">

We will solve this puzzle by encoding it as a *constrant satisfaction problem*.
- The set of values is $\{1,\cdots,8\}$.  These values are interpreted as columns.
- We will use 8 variables $Q_1$, $\cdots$, $Q_8$.  If $Q_r = c$, then the interpretation 
  is that there is a queen at the position $(r, c)$, i.e. the queen in row $r$ is 
  placed in column $c$. 
- The condition 
  $$ \bigl\{ \texttt{Q}_i \not= \texttt{Q}_j \;\bigm|\; 
      i \in \{1,\cdots,8\} \wedge j \in \{1,\cdots,8\} \wedge j < i \bigr\} 
  $$
  specifies that there is at most on queen in a given column.
- The condition
  $$ \bigl\{ |\texttt{Q}_i - \texttt{Q}_j| \not= |i - j| \;\bigm|\; 
     i \in \{1,\cdots,8\} \wedge j \in \{1,\cdots,8\} \wedge j < i \bigr\}
  $$
  specifies that there is at most one queen in a given diagonal.

We use the backtracking constraint solver from the notebook `02-Backtracking-Constraint-Solver.ipynb`, which has been loaded above.

The function `queensCSP(n)` takes the size `n` of the board as input and returns
a *constraint satisfaction problem* that encodes the *n-queens problem*.

As Java does not support default values for parameters, we define a second version of `queensCSP` that takes no parameters and uses the board size 8.

In [ ]:
CSP queensCSP(int n) {
    var Vars    = range(1, n).stream().map(i -> "Q" + i).toList();
    var Values  = range(1, n);
    var Constrs = range(1, n).stream().flatMap(
        i -> range(i + 1, n).stream().flatMap(j -> Stream.of("Q" + i + " != Q" + j,
                                                             "Math.abs(Q" + i + " - Q" + j + ") != " + (j - i))))
        .toList();
    return new CSP(Vars, Values, Constrs);
}

CSP queensCSP() {
    return queensCSP(8);
}

Next, we create a *CSP* that encodes the *8-queens-puzzle*.

In [ ]:
var csp = queensCSP();
csp

This *CSP* has 8 variables and 56 constraints.  When we solved the same problem using propositional logic and the *Davis-Putnam algorithm*, we had used 64 propositional variables and 512 clauses.

In [ ]:
List.of(csp.variables().size(), csp.constraints().size())

Solving the problem takes only a few milliseconds.  Most of the time is spent compiling the constraints.

In [ ]:
long start   = System.currentTimeMillis();
var solution = solve(csp);
System.out.println("solution: " + (System.currentTimeMillis() - start) + " ms");
solution

The function `showSolution(solution, n, width)` takes a map `solution` that contains a variable assignment representing a solution to the n-queens puzzle. It displays this solution on a chess board.

* `solution` is a map mapping the variables $\texttt{Q}_i$ to numbers.
  If $\texttt{solution.get("Q}_i\texttt{")} = k$, then the queen in column $i$ is placed in row $k$.
* `n` specifies the size of the board (default is 8).
* `width` specifies the display width of the board (default is "50%").

In [ ]:
void showSolution(Map<String, Integer> solution, int n, String width) {
    String[][] boardArray = new String[n][n];
    for (String[] row : boardArray) {
        Arrays.fill(row, "");
    }
    for (int col = 1; col <= n; col++) {
        Integer row = solution.get("Q" + col);
        if (row != null && row - 1 < n && col - 1 < n) {
            boardArray[row - 1][col - 1] = "♕";
        }
    }
    var html = new StringBuilder();
    html.append("""
        <div style="display: grid; grid-template-columns: repeat(%d, 1fr); width: %s;
                    aspect-ratio: 1/1; border: 2px solid #8b4513;">""".formatted(n, width));
    for (int row = 0; row < n; row++) {
        for (int col = 0; col < n; col++) {
            String piece       = boardArray[row][col];
            String bgColor     = (row + col) % 2 == 0 ? "#ffce9e" : "#d18b47";
            String cellContent = "";
            if (!piece.isEmpty()) {
                cellContent = """
                    <svg viewBox="0 0 100 100" style="width: 80%; height: 80%; display: block;">
                        <text x="50%" y="55%" font-size="90" text-anchor="middle"
                              dominant-baseline="middle" fill="black">""" + piece + """
                    </text>
                    </svg>""";
            }
            html.append("""
                <div style="display: flex; align-items: center; justify-content: center;
                            background-color: %s; overflow: hidden;">%s</div>""".formatted(bgColor, cellContent));
        }
    }
    html.append("</div>");
    display(html.toString(), "text/html");
}

void showSolution(Map<String, Integer> solution, int n) {
    showSolution(solution, n, "50%");
}

In [ ]:
showSolution(solution, 8);

If we would use a brute force approach that checks all permutations, we would have to check $8! = 40\,320$ different configurations.  

In [ ]:
(long) Math.pow(8, 8)